In [41]:
# ** This cell is needed since we are not in the src directory 
import sys 
import os
# Add the src/ directory to the Python path
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "../src/")))

ROOT_DIR = ".."
SRC_DIR = ROOT_DIR + "/src"

import sys
sys.path.append("/Users/admin/eeg-ds004504/")


In [42]:
from config_handler import initiate_config, load_config

In [43]:
initiate_config()

{'data_path': '/Users/admin/eeg-ds004504',
 'derivatives': True,
 'freqBands': {'Alpha': [8, 12],
  'Beta': [12, 30],
  'Delta': [0.5, 4],
  'Theta': [4, 8],
  'custom1': [9, 11]},
 'method': 'welch',
 'stepSize': 3.0,
 'windowLength': 3.0}

In [44]:
print(load_config())

{'data_path': '/Users/admin/eeg-ds004504', 'derivatives': True, 'freqBands': {'Alpha': [8, 12], 'Beta': [12, 30], 'Delta': [0.5, 4], 'Theta': [4, 8], 'custom1': [9, 11]}, 'method': 'welch', 'stepSize': 3.0, 'windowLength': 3.0}


In [45]:
from pyspark.sql import SparkSession, DataFrame
from pyspark.sql.functions import pandas_udf, PandasUDFType
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, ArrayType, MapType
import pandas as pd
from functools import reduce

In [46]:
# Check if there's an active Spark context and stop it
from pyspark import SparkContext
if SparkContext._active_spark_context:
    print("Stopping existing Spark context...")
    SparkContext._active_spark_context.stop()
    print("Previous Spark context stopped successfully")

In [47]:
import os
from pyspark.sql import SparkSession

# Set Java options for the JVM running Spark
# -Xmx12g : Sets the maximum heap size to 12GB
# -Xms4g : Sets the initial heap size to 4GB to avoid resizing overhead
os.environ["_JAVA_OPTIONS"] = "-Xmx12g -Xms4g"

# Build Spark session with memory, parallelism, and network settings
spark = (
    SparkSession.builder 
    # Application name shown in Spark UI
    .appName("EEG_Analysis") 

    # Use all available logical cores or specify a number
    # "local[*]" uses all available cores, "local[12]" limits to 12 threads
    .config("spark.master", "local[12]") \

    # Executor memory: how much memory each Spark worker can use
    .config("spark.executor.memory", "8g") \

    # Driver memory: memory available to the Spark driver (main Python process)
    .config("spark.driver.memory", "8g") \

    # Number of shuffle partitions (e.g., after groupBy, join, etc.)
    # Lower this in local mode to reduce overhead (default is 200)
    .config("spark.sql.shuffle.partitions", "12") \

    # Default number of partitions in operations like parallelize
    .config("spark.default.parallelism", "12") \

    # Maximum size (in MB) allowed for any RPC message (e.g., large UDF closures or data broadcasts)
    .config("spark.rpc.message.maxSize", "256") \
    
    # to not reach 100% CPU utilizatoin and get stuck
    .config("spark.master", "local[8]")

    # Required for avoiding binding issues on some MacOS environments
    .config("spark.driver.bindAddress", "127.0.0.1") \
    .config("spark.driver.host", "127.0.0.1") 
    .getOrCreate()
)
# Enable Apache Arrow for pandas UDF performance boost
spark.conf.set("spark.sql.execution.arrow.pyspark.enabled", "true")


print("New Spark session created successfully")

New Spark session created successfully


25/09/07 01:34:35 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [48]:
from populate_schemas import load_subjects_df, extract_features_udtf
from feature_extraction import processEpoch, processSub
from schema_definition import get_feature_schema, get_subject_schema

sc = spark.sparkContext # we pass udf/udtf's (user defind functions and user defined table functions) to spark so it can access them

# Making all necessary modules available to spark
try:
    sc.addPyFile(os.path.join(SRC_DIR, "feature_extraction.py"))
    print("Added feature_extraction.py to the pyspark context")
    sc.addPyFile(os.path.join(SRC_DIR, "feature_extraction_helper.py"))
    print("Added feature_extraction_helper.py to the pyspark context")
    sc.addPyFile(os.path.join(SRC_DIR, "preprocess_sets.py"))
    print("Added preprocess_sets to the pyspark context")
    sc.addPyFile(os.path.join(SRC_DIR, "schema_definition.py"))
    print("Added schema_definition.py to the pyspark context")
    sc.addPyFile(os.path.join(SRC_DIR, "config_handler.py"))
    print("Added config_handler.py to the pyspark context")
except Exception as e:
    print(f"Error adding files to SparkContext: {e}")

Added feature_extraction.py to the pyspark context
Added feature_extraction_helper.py to the pyspark context
Added preprocess_sets to the pyspark context
Added schema_definition.py to the pyspark context
Added config_handler.py to the pyspark context


# Testing for 1 subject

In [49]:
subject_df = load_subjects_df(spark) #this is the .tsv with the information of all the participantsfname
subject_df = subject_df.repartition(200, "SubjectID")  # tweak 200 based on cluster resources

In [51]:
%%time
# we need to give the path of our  data directory to process the EEG data from
from preprocess_sets import get_data_path

# set_data_path("/Users/user/eeg-ds004504") !!! this doesn't work! so we need to do it manually in preprocess_sets.py! or else won't work!
print(get_data_path())

#Example below is how to get a single subject  and extract its features
sub1 = (
    subject_df
    .filter((subject_df.SubjectID == "sub-003"))
    .groupBy("SubjectID")
    .apply(extract_features_udtf)
)
sub1.persist()
print("3 tables, one for each layer of abraction, specific --> general (band, electrode, epoch) ")
sub1.filter((sub1.table_type=="band")).show(3)
sub1.filter((sub1.table_type=="electrode")).show(3)
sub1.filter((sub1.table_type=="epoch")).show(3)
print("total epochs in this is")
print(sub1.select("EpochID").distinct().count())

/Users/admin/eeg-ds004504
3 tables, one for each layer of abraction, specific --> general (band, electrode, epoch) 


25/09/07 01:35:01 WARN CacheManager: Asked to cache already cached data.


+---------+-------+---------+--------+---------------+------------+----------+
|SubjectID|EpochID|Electrode|WaveBand|    FeatureName|FeatureValue|table_type|
+---------+-------+---------+--------+---------------+------------+----------+
|  sub-003|   ep-0|      Fp1|   Alpha|          Power| 0.008017266|      band|
|  sub-003|   ep-0|      Fp1|   Alpha|SpectralEntropy|   2.2665515|      band|
|  sub-003|   ep-0|      Fp1|   Alpha| HjorthMobility| 0.045576047|      band|
+---------+-------+---------+--------+---------------+------------+----------+
only showing top 3 rows

+---------+-------+---------+--------+---------------+------------+----------+
|SubjectID|EpochID|Electrode|WaveBand|    FeatureName|FeatureValue|table_type|
+---------+-------+---------+--------+---------------+------------+----------+
|  sub-003|   ep-0|      Fp1|    NULL|     TotalPower| 0.011235955| electrode|
|  sub-003|   ep-0|      Fp1|    NULL|    TotalEnergy|  0.10781828| electrode|
|  sub-003|   ep-0|      Fp

In [11]:
sub1.printSchema()

root
 |-- SubjectID: string (nullable = false)
 |-- EpochID: string (nullable = false)
 |-- Electrode: string (nullable = true)
 |-- WaveBand: string (nullable = true)
 |-- FeatureName: string (nullable = true)
 |-- FeatureValue: float (nullable = true)
 |-- table_type: string (nullable = true)



In [12]:
sub1.show()

+---------+-------+---------+--------+----------------+------------+----------+
|SubjectID|EpochID|Electrode|WaveBand|     FeatureName|FeatureValue|table_type|
+---------+-------+---------+--------+----------------+------------+----------+
|  sub-003|   ep-0|      Fp1|    NULL|      TotalPower| 0.011235955| electrode|
|  sub-003|   ep-0|      Fp1|    NULL|     TotalEnergy|  0.10781828| electrode|
|  sub-003|   ep-0|      Fp1|    NULL| SpectralEntropy|   4.1740084| electrode|
|  sub-003|   ep-0|      Fp1|    NULL|  HjorthActivity|9.705875E-10| electrode|
|  sub-003|   ep-0|      Fp1|    NULL|  HjorthMobility| 0.045576047| electrode|
|  sub-003|   ep-0|      Fp1|    NULL|HjorthComplexity|   5.9060574| electrode|
|  sub-003|   ep-0|      Fp1|    NULL|     HjorthIndex|   1108.8796| electrode|
|  sub-003|   ep-0|      Fp1|   Alpha|           Power| 0.008017266|      band|
|  sub-003|   ep-0|      Fp1|   Alpha| SpectralEntropy|   2.2665515|      band|
|  sub-003|   ep-0|      Fp1|   Alpha|  

In [15]:
subject_df.show(65)

+---------+-----+
|SubjectID|Group|
+---------+-----+
|  sub-067|    F|
|  sub-058|    C|
|  sub-081|    F|
|  sub-057|    C|
|  sub-020|    A|
|  sub-033|    A|
|  sub-037|    C|
|  sub-055|    C|
|  sub-077|    F|
|  sub-083|    F|
|  sub-012|    A|
|  sub-002|    A|
|  sub-062|    C|
|  sub-011|    A|
|  sub-060|    C|
|  sub-069|    F|
|  sub-016|    A|
|  sub-082|    F|
|  sub-049|    C|
|  sub-073|    F|
|  sub-085|    F|
|  sub-017|    A|
|  sub-025|    A|
|  sub-053|    C|
|  sub-059|    C|
|  sub-078|    F|
|  sub-084|    F|
|  sub-021|    A|
|  sub-064|    C|
|  sub-018|    A|
|  sub-029|    A|
|  sub-079|    F|
|  sub-074|    F|
|  sub-001|    A|
|  sub-054|    C|
|  sub-039|    C|
|  sub-063|    C|
|  sub-065|    C|
|  sub-009|    A|
|  sub-061|    C|
|  sub-050|    C|
|  sub-041|    C|
|  sub-052|    C|
|  sub-022|    A|
|  sub-046|    C|
|  sub-036|    A|
|  sub-088|    F|
|  sub-072|    F|
|  sub-004|    A|
|  sub-027|    A|
|  sub-006|    A|
|  sub-019|    A|
|  sub-030

# Multi-threaded Version

In [29]:
group_a_spark_df = (
    subject_df
    .filter(subject_df.Group == "A")
    .groupBy("SubjectID")
    .apply(extract_features_udtf)
)

group_c_spark_df = (
    subject_df
    .filter(subject_df.Group == "C")
    .groupBy("SubjectID")
    .apply(extract_features_udtf)
)


group_f_spark_df = (
    subject_df
    .filter(subject_df.Group == "F")
    .groupBy("SubjectID")
    .apply(extract_features_udtf)
)


# Forcing single-thread (if multithread version doesn't work)

In [24]:
%%time



group_a_subjects = (
    subject_df.filter(subject_df.Group == "A")
    .select("SubjectID")
    .distinct()
    .rdd.flatMap(lambda r: r)
    .collect()
)

group_c_subjects = (
    subject_df.filter(subject_df.Group == "C")
    .select("SubjectID")
    .distinct()
    .rdd.flatMap(lambda r: r)
    .collect()
)

group_f_subjects = (
    subject_df.filter(subject_df.Group == "F")
    .select("SubjectID")
    .distinct()
    .rdd.flatMap(lambda r: r)
    .collect()
)

results_a = []
for subj in group_a_subjects:
    print(f"\n=== Processing A: {subj} ===")
    try:
        df = (
            subject_df
            .filter(subject_df.SubjectID == subj)
            .groupBy("SubjectID")
            .apply(extract_features_udtf)
        ).persist()
        
        _ = df.count()  # ✅ Force execution right now
        results_a.append(df)
        print(f"✅ Finished A: {subj}")
    except Exception as e:
        print(f"[ERROR] Subject A {subj} failed: {e}")

results_c = []
for subj in group_c_subjects:
    print(f"\n=== Processing C: {subj} ===")
    try:
        df = (
            subject_df
            .filter(subject_df.SubjectID == subj)
            .groupBy("SubjectID")
            .apply(extract_features_udtf)
        ).persist()
        
        _ = df.count()  # ✅ Force execution right now
        results_c.append(df)
        print(f"✅ Finished C: {subj}")
    except Exception as e:
        print(f"[ERROR] Subject C {subj} failed: {e}")

results_f = []
for subj in group_f_subjects:
    print(f"\n=== Processing F: {subj} ===")
    try:
        df = (
            subject_df
            .filter(subject_df.SubjectID == subj)
            .groupBy("SubjectID")
            .apply(extract_features_udtf)
        ).persist()
        
        _ = df.count()  # ✅ Force execution right now
        results_f.append(df)
        print(f"✅ Finished C: {subj}")
    except Exception as e:
        print(f"[ERROR] Subject C {subj} failed: {e}")


result_group_a = reduce(DataFrame.unionByName, results_a)
result_group_c = reduce(DataFrame.unionByName, results_c)
result_group_f = reduce(DataFrame.unionByName, results_f)

print(f"\n✅ All Group A subjects done ({len(results_a)} total)")
print(f"✅ All Group C subjects done ({len(results_c)} total)")






=== Processing A: sub-020 ===
✅ Finished A: sub-020

=== Processing A: sub-033 ===


25/09/07 00:25:46 WARN CacheManager: Asked to cache already cached data.
25/09/07 00:25:46 WARN CacheManager: Asked to cache already cached data.


✅ Finished A: sub-033

=== Processing A: sub-012 ===
✅ Finished A: sub-012

=== Processing A: sub-002 ===


25/09/07 00:25:46 WARN CacheManager: Asked to cache already cached data.
25/09/07 00:25:46 WARN CacheManager: Asked to cache already cached data.


✅ Finished A: sub-002

=== Processing A: sub-011 ===
✅ Finished A: sub-011

=== Processing A: sub-016 ===


...

✅ Finished C: sub-071

=== Processing F: sub-066 ===
✅ Finished C: sub-066

=== Processing F: sub-086 ===


25/09/07 00:25:56 WARN CacheManager: Asked to cache already cached data.
25/09/07 00:25:56 WARN CacheManager: Asked to cache already cached data.


✅ Finished C: sub-086

=== Processing F: sub-076 ===
✅ Finished C: sub-076


25/09/07 00:25:56 WARN CacheManager: Asked to cache already cached data.



✅ All Group A subjects done (36 total)
✅ All Group C subjects done (29 total)
CPU times: user 446 ms, sys: 591 ms, total: 1.04 s
Wall time: 12.2 s


In [27]:
result_group_f.show(1)

+---------+-------+---------+--------+-----------+------------+----------+
|SubjectID|EpochID|Electrode|WaveBand|FeatureName|FeatureValue|table_type|
+---------+-------+---------+--------+-----------+------------+----------+
|  sub-067|   ep-0|      Fp1|    NULL| TotalPower| 0.011235955| electrode|
+---------+-------+---------+--------+-----------+------------+----------+
only showing top 1 row



In [30]:
# # Use persist to cache the results in memory as we will probably reference this often, so might as well keep it in ram!
result_group_a = group_a_spark_df.persist()
result_group_c = group_c_spark_df.persist()
result_group_f = group_f_spark_df.persist()

# Trigger execution with actions
count_a = result_group_a.count()
count_c = result_group_c.count()
count_f = result_group_f.count()

print(f"Processed {count_a} records for Alzheimer's group")
print(f"Processed {count_c} records for Control group")
print(f"Processed {count_f} records for Control group")


25/09/07 00:27:33 WARN CacheManager: Asked to cache already cached data.
25/09/07 00:27:33 WARN CacheManager: Asked to cache already cached data.
Config not found in feature_extraction.py==============>        (168 + 8) / 200]
Config using in feature Extraction.py {'data_path': '/Users/admin/eeg-ds004504', 'derivatives': True, 'freqBands': {'Alpha': [8, 12], 'Beta': [12, 30], 'Delta': [0.5, 4], 'Theta': [4, 8], 'custom1': [9, 11]}, 'method': 'welch', 'stepSize': 3.0, 'windowLength': 3.0}
[START] sub-021
processSub sub-021
processSub: derivatives True
processSub: windowLength 3.0
processSub: windowLength 3.0
Config not found in feature_extraction.py
Config using in feature Extraction.py {'data_path': '/Users/admin/eeg-ds004504', 'derivatives': True, 'freqBands': {'Alpha': [8, 12], 'Beta': [12, 30], 'Delta': [0.5, 4], 'Theta': [4, 8], 'custom1': [9, 11]}, 'method': 'welch', 'stepSize': 3.0, 'windowLength': 3.0}
[START] sub-012
processSub sub-012
processSub: derivatives True
processSub: w

Processed 5784881 records for Alzheimer's group
Processed 3289358 records for Control group


NameError: name 'count_f' is not defined

In [31]:
alz_df_spark = result_group_a 
cntrl_df_spark = result_group_c
ftd_df_spark = result_group_f

In [32]:
result_group_a.columns

['SubjectID',
 'EpochID',
 'Electrode',
 'WaveBand',
 'FeatureName',
 'FeatureValue',
 'table_type']

In [34]:
print(result_group_a.select("table_type", "FeatureName").groupBy("table_type").count().show())
print(result_group_c.select("table_type", "FeatureName").groupBy("table_type").count().show())
print(result_group_f.select("table_type", "FeatureName").groupBy("table_type").count().show())


+----------+-------+
|table_type|  count|
+----------+-------+
| electrode|1255121|
|      band|4482575|
|     epoch|  47185|
+----------+-------+

None
+----------+-------+
|table_type|  count|
+----------+-------+
| electrode|1031149|
|      band|3682675|
|     epoch|  38765|
+----------+-------+

None
+----------+-------+
|table_type|  count|
+----------+-------+
| electrode| 713678|
|      band|2548850|
|     epoch|  26830|
+----------+-------+

None


In [35]:
print(result_group_a.select("table_type", "FeatureName").groupBy("table_type").count().show())

+----------+-------+
|table_type|  count|
+----------+-------+
| electrode|1255121|
|      band|4482575|
|     epoch|  47185|
+----------+-------+

None


In [36]:
from datetime import datetime
import os
from config_handler import load_config

def create_database_log(filename, config, timestamp, df=None):
    """
    Creates a log file with information about the saved dataframe.
    
    Parameters:
    - df: The Spark DataFrame being saved
    - filename: The name of the saved pkl file
    - config: Configuration dictionary from load_config()
    - timestamp: Timestamp string for the log entry
    """
    # Count unique subjects
    try:
        unique_subjects = df.select("SubjectID").distinct().count()
    except:
        unique_subjects = -1
    
    # Create log entry
    log_entry = [
        f"=== DATABASE LOG ENTRY: {timestamp} ===",
        f"Saved file: {filename}",
        f"Unique subjects: {unique_subjects}",
        "\nConfiguration:"
    ]
    
    # Add configuration details
    for key, value in config.items():
        if key == "freqBands":
            log_entry.append(f"  {key}:")
            for band, freq_range in value.items():
                log_entry.append(f"    {band}: {freq_range}")
        else:
            log_entry.append(f"  {key}: {value}")
    
    log_entry.append("\n")
    
    # Write to log file
    with open("databaselog.txt", "a") as f:
        f.write("\n".join(log_entry))
    
    print(f"Log entry written to databaselog.txt")

In [37]:
from datetime import datetime
from config_handler import load_config

# Generate a timestamp like "Apr14_2230"
timestamp = datetime.now().strftime("%b%d_%H%M")

# Filenames with timestamp
alz_filename = f"features_alz_extra_features_{timestamp}"
cntrl_filename = f"features_cntrl_extra_features_{timestamp}"
ftd_filename = f"features_ftd_extra_features_{timestamp}"

# Save with timestamp in filename
# group_a_pandas_df.to_pickle(f"{alz_filename}.pkl")
# group_c_pandas_df.to_pickle(f"{cntrl_filename}.pkl")

# Get configuration once
config = load_config()

# Create log entries for both files
create_database_log(alz_filename, config, timestamp)
create_database_log(cntrl_filename, config, timestamp)
create_database_log(ftd_filename, config, timestamp)

Log entry written to databaselog.txt
Log entry written to databaselog.txt
Log entry written to databaselog.txt


In [38]:
# Save alz_df_spark to Parquet
alz_df_spark.write.mode("overwrite").parquet(f"{alz_filename}.parquet")

# Save cntrl_df_spark to Parquet
cntrl_df_spark.write.mode("overwrite").parquet(f"{cntrl_filename}.parquet")

ftd_df_spark.write.mode("overwrite").parquet(f"{ftd_filename}.parquet")

# checking the difference between pkl files

In [57]:
# This is how we would load the .pkl's back in 
# Step 1: Load back into pandas


group_a_parquet_df_loaded = spark.read.parquet(f"{alz_filename}.parquet")
group_c_parquet_df_loaded = spark.read.parquet(f"{cntrl_filename}.parquet")


In [58]:
type(group_a_parquet_df_loaded)

pyspark.sql.dataframe.DataFrame

In [54]:
if group_a_parquet_df_loaded.exceptAll(group_a_spark_df).isEmpty(): 
    print("Correctly pkl'd and correcfly loaded into pyspark object")

25/04/24 08:24:25 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
25/04/24 08:24:25 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
25/04/24 08:24:25 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
25/04/24 08:24:25 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
25/04/24 08:24:25 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
25/04/24 08:24:25 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
25/04/24 08:24:26 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
25/04/24 08:24:27 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
[Stage 1095:================>                                   

Correctly pkl'd and correcfly loaded into pyspark object


In [ ]:
group_c_spark_df_loaded.select("SubjectID").distinct().orderBy("SubjectID").show(truncate=False)

In [39]:
spark.stop()